In [1]:

import numpy as np 
import pandas as pd
import os
from PIL import Image
import torch
import  matplotlib.pyplot as plt

DATA_PATH = "E:/ML/UBC"


In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [3]:
trainDf = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
allIds = trainDf["image_id"].unique()
trainDf.head()

,image_id,label,image_width,image_height,is_tma
0,4,HGSC,23785,20008,False
1,66,LGSC,48871,48195,False
2,91,HGSC,3388,3388,True
3,281,LGSC,42309,15545,False
4,286,EC,37204,30020,False


In [4]:
trainListIndexed = trainDf.set_index("image_id")

In [5]:
trainListWang = pd.read_csv(os.path.join(DATA_PATH, "Wang Dataset - all.csv"))
trainListWang.set_index("Image ID", inplace=True)
trainListWang.head()


,No,Patient ID,Treatment effect,Image No.,CA-125 before,CA-125 after,Diagnosis
Image ID,,,,,,,
1612595C,1,2909711,effective,1612595C.svs,91.52,16.72,PSPC (Peritoneal serous papillary carcinoma)
1702510F,2,2959832,effective,1702510F.svs,H>200,29.4,UC
1717035D,3,2983362,effective,1717035D.svs,77.74,27.15,CC
500203D,4,801250,effective,500203D.svs,H>200,18.39,CC
1832548A,5,2188987,effective,1832548A.svs,87.38,8.04,PsC


In [6]:
IMG_SIZE = (384, 384)
eps=1e-12

def readImage(path, skipResize=False):
    data = Image.open(path).convert("RGB")
        
    w, h = data.width, data.height
    # centerWindow = data[w//4:3*w//4, h//4:3*h//4]
    # medValue = np.median(data)

    #Center crop
    data = np.array(data)
    if w>h:
        diff = w-h
        data = data[diff//2:diff//2+h, :, :]
    if h>w:
        diff = h-w
        data = data[:, diff//2:diff//2+w, :]

    # data = data - np.min(data)
    # data = data * 1.0/(np.max(data)+eps)

    w, h, c = data.shape[0], data.shape[1], data.shape[2]

    # resize
    if not skipResize:
        if not (w == IMG_SIZE[0] and h == IMG_SIZE[1]):
            data = np.array(Image.fromarray(data).resize(IMG_SIZE))
    
    # data = data/(np.max(data)+eps) * 2 - 1

    # data = (data * 255).astype(np.uint8)
    return data



In [7]:
from sklearn.preprocessing import LabelEncoder
import warnings
import pickle 

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with open('./LabelEncoder.pkl', 'rb') as f:
        enc = pickle.load(f) 
    print(enc.classes_)
    print(enc.transform(["LGSC"]))

['CC' 'EC' 'HGSC' 'LGSC' 'MC' 'Other']
[3]


In [8]:
# import timm
# from torchvision.models.feature_extraction import get_graph_node_names
# from torchvision.models.feature_extraction import create_feature_extractor
# import torchvision

# model = timm.create_model('edgenext_base', pretrained=True, num_classes=len(enc.classes_))
# print(model)


# train_nodes, eval_nodes = get_graph_node_names(model)
# print(eval_nodes)

# checkpoint = torch.load(os.path.join("./", "edgenextBase_epoch_8.pt"), map_location=device)
# model.load_state_dict(checkpoint['model_state_dict'])
# model.to(device)
# model.eval()


# return_nodes = {
#     'global_pool.flatten': 'features',
# }

# model = create_feature_extractor(model, return_nodes=return_nodes)

In [9]:
import timm
from torchvision.models.feature_extraction import get_graph_node_names
from torchvision.models.feature_extraction import create_feature_extractor
import torchvision

model = timm.create_model('edgenext_base', pretrained=True, num_classes=len(enc.classes_))
print(model)

checkpoint = torch.load(os.path.join("./", "edgenextBase_epoch_8.pt"), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()


c:\Users\Manuel\anaconda3\envs\torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EdgeNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 80, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((80,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): EdgeNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvBlock(
          (conv_dw): Conv2d(80, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=80)
          (norm): LayerNorm((80,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=80, out_features=320, bias=True)
            (act): GELU(approximate='none')
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Linear(in_features=320, out_features=80, bias=True)
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (drop_path): Identity()
        )
        (1): ConvBlock(
          (conv_dw): Conv2d(80, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=80)
          (norm): Layer

EdgeNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 80, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((80,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): EdgeNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvBlock(
          (conv_dw): Conv2d(80, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=80)
          (norm): LayerNorm((80,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=80, out_features=320, bias=True)
            (act): GELU(approximate='none')
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Linear(in_features=320, out_features=80, bias=True)
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (drop_path): Identity()
        )
        (1): ConvBlock(
          (conv_dw): Conv2d(80, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=80)
          (norm): Layer

In [10]:

activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

h = model.head.flatten.register_forward_hook(get_activation('global_pool'))

x = torch.randn((1,3,384,384)).to(device)
output = model(x)
print(activation["global_pool"].shape)
# h.remove()

torch.Size([1, 584])


In [11]:
# testIn = torch.randn((1,3,256,256)).to(device)
# out = model(testIn)
# out["features"].shape

In [12]:
from torchvision.transforms import v2
from tqdm import tqdm

transformsVal = v2.Compose([
    v2.ToDtype(torch.float32),
    # v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

X=[]
y=[]
tma=[]

for tid in tqdm(allIds, smoothing=0.05):
    imFolder = os.path.join(DATA_PATH, "trainProcessed\\patches7x7", str(tid))
    imArray = []
    targets = trainListIndexed.loc[tid]
    label = np.array([targets["label"]])
    isTMA = np.array([targets["is_tma"]])
    # if not isTMA[0] and np.random.random()<0.93:
    #     continue
    encLabel = enc.transform(label)

    for root, dirs, files in os.walk(imFolder):
        for f in files:
            if f.endswith(".png"):
                data = readImage(os.path.join(root, f))
                imArray.append(data)
    imArray = np.swapaxes(np.array(imArray), 1,-1).astype(np.float32)/255.0
    imArray = transformsVal(torch.Tensor(imArray)).to(device)

    out = model(imArray)
    # featVect = out["features"]
    featVect = activation["global_pool"]
    # featVect = torch.max(featVect, 0)
    # X.append(featVect.values.detach().cpu().numpy())
    
    X.append(featVect.detach().cpu().numpy())
    y.append(encLabel[0])


# for tid in tqdm(list(trainListWang.index)):
#     imFolder = os.path.join(DATA_PATH, "PKG-OBR_Patches", str(tid))
#     targets = trainListWang.loc[tid]
#     label = np.array([targets["Diagnosis"]])[0]
#     if pd.isna(label):
#         continue
#     if "UC" in label:
#         label = np.array(["Other"])
#         encLabel = enc.transform(label)
#     elif "CC" in label:
#         label = np.array(["CC"])
#         encLabel = enc.transform(label)
#     elif "EmAC" in label:
#         label = np.array(["EC"])
#         encLabel = enc.transform(label)
#     elif "MC" in label:
#         label = np.array(["MC"])
#         encLabel = enc.transform(label)
#     elif "PSPC" in label or "PsC" in label or "PSC" in label or "Psc" in label:
#         label = np.array(["HGSC"])
#         encLabel = enc.transform(label)
#     else:
#         print("invalid label for PKG-OBR", label)
#     imArray = []
#     for root, dirs, files in os.walk(imFolder):
#         for f in files:
#             if f.endswith(".png"):
#                 data = readImage(os.path.join(root, f))
#                 imArray.append(data)
#     if len(imArray)==0:
#         print("empty array", imFolder)
#         continue
#     imArray = np.swapaxes(np.array(imArray), 1,-1).astype(np.float32)/255.0
#     imArray = transformsVal(torch.Tensor(imArray)).to(device)

#     out = model(imArray)
#     featVect = out["features"]
#     featVect = torch.max(featVect, 0)

#     X.append(featVect.values.detach().cpu().numpy())
#     y.append(encLabel[0])


c:\Users\Manuel\anaconda3\envs\torch\lib\site-packages\torchvision\datapoints\__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
c:\Users\Manuel\anaconda3\envs\torch\lib\site-packages\torchvision\transforms\v2\__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you m

100%|██████████| 538/538 [1:07:40<00:00,  7.55s/it]


In [13]:
X = np.array(X)
y = np.array(y)
X.shape

(538, 49, 584)

In [15]:

with open(os.path.join(DATA_PATH,"./FeatureDataEdgeNext.pkl"), "wb") as f:
        pickle.dump(zip(X,y), f)